# 3. Эксперименты с денойзингом

Проверяем три подхода:
1. UNet + L1 loss (обучение на парах noisy -> clean)
2. UNet + Noise2Void (без пар)
3. ResNet + Noise2Void

**Вывод:** лучший результат дал heuristic-метод из ноутбука 02 — он и используется дальше
Веса всех экспериментов сохранены в `models/`

In [ ]:
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader
from torchvision.transforms import transforms
import matplotlib.pyplot as plt

from src.config import IMAGES_DIR, MODELS_DIR, FIGURES_DIR, SEED
from src.datasets import NoiseClearDataset
from src.models import UNetForDenoise, DenoiseResNet
from src.train import train_denoiser_unet, train_noise2void
from src.utils import tensor_to_pil, set_seed


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = transforms.Compose([transforms.ToTensor()])
dataset = NoiseClearDataset(str(IMAGES_DIR), transform=train_transform)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
unet_path = MODELS_DIR / "denoiser_weights.pth"
unet = UNetForDenoise().to(device)

if unet_path.exists():
    unet.load_state_dict(torch.load(unet_path, map_location=device))
    print("Веса UNet загружены")
else:
    # train_denoiser_unet(unet, loader, epochs=1, device=device)
    # torch.save(unet.state_dict(), unet_path)
    pass

In [ ]:
# Визуализация результата
img_path = IMAGES_DIR / "2.jpg"
import numpy as np
from PIL import Image
original = Image.open(img_path).convert("RGB")
x = train_transform(original).unsqueeze(0).to(device)
unet.eval()
with torch.no_grad():
    denoised = unet(x).squeeze(0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(tensor_to_pil(denoised)); axes[1].set_title("UNet"); axes[1].axis("off")
plt.tight_layout()
plt.show()